# Laboratorio 04 — Pipeline RAG básico sobre documentos propios
### Python AI Developer 2026 · Capítulo 3: RAG

**Objetivo:** construir un pipeline RAG completo — ingestar una colección de documentos, dividirlos en chunks, indexarlos en una base vectorial, y usarlos para responder preguntas con un LLM.

**Por qué una colección y no un solo documento:** con un solo documento se podría pegar el texto completo en el prompt y obtener el mismo resultado. RAG resuelve el problema de escala — buscar en cientos de documentos sin enviarlos todos al modelo. Con varios documentos, el retriever tiene que elegir cuál es relevante para cada pregunta, y ahí se ve su valor real.

**Duración estimada:** 90 min

**Setup:**
```bash
uv add langchain langchain-core langchain-text-splitters langchain-community
uv add langchain-chroma langchain-anthropic langchain-huggingface
uv add chromadb sentence-transformers anthropic python-dotenv
```

---
## Parte 1 — El problema que resuelve RAG

Los LLMs tienen dos limitaciones fundamentales:
1. **Fecha de corte:** no conocen información posterior a su entrenamiento
2. **Alucinación:** cuando no saben algo, inventan respuestas plausibles pero incorrectas

RAG (Retrieval-Augmented Generation) resuelve ambas: antes de generar, el modelo recupera documentos relevantes y los usa como contexto verificable.

```
Sin RAG:  pregunta → LLM → respuesta (puede ser inventada)
Con RAG:  pregunta → retriever → documentos → LLM → respuesta basada en evidencia
```

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

assert os.getenv("ANTHROPIC_API_KEY"), "Falta ANTHROPIC_API_KEY en .env"
print("Credenciales OK")

Credenciales OK


---
## Parte 2 — Crear la colección de documentos

Usamos 5 documentos internos de una empresa ficticia (TechCorp). Cada uno cubre un tema distinto. El documento de RRHH actúa como distractor para preguntas técnicas — el retriever debe ignorarlo cuando no es relevante.

In [2]:
import os

documentos = {
    "manual_servidores.txt": "MANUAL DE INFRAESTRUCTURA — TechCorp Perú S.A.C.\n\nSERVIDORES DE PRODUCCIÓN\nTechCorp opera 3 servidores de producción en el datacenter de Lima:\n- Server-PROD-01: aplicación principal, 32 cores, 128 GB RAM\n- Server-PROD-02: base de datos PostgreSQL 15, 64 cores, 256 GB RAM\n- Server-PROD-03: servicios de cache Redis y cola de mensajes RabbitMQ\nEl uptime mínimo garantizado es 99.9% mensual. Cualquier caída debe reportarse al equipo de DevOps en un máximo de 5 minutos mediante el canal #incidentes en Slack.\n\nPROCEDIMIENTO DE BACKUP\nLos backups se ejecutan diariamente a las 2:00 AM hora de Lima:\n- Backup completo: todos los domingos\n- Backup incremental: lunes a sábado\n- Retención: 30 días para backups diarios, 12 meses para backups semanales\n- Almacenamiento: bucket S3 en región us-east-1 con cifrado AES-256\n\nCAPACIDAD\nEl umbral de alerta de CPU es 80%. El sistema escala automáticamente agregando instancias EC2 t3.large. El máximo de instancias de escalado automático es 5.",

    "manual_seguridad.txt": "POLÍTICAS DE SEGURIDAD — TechCorp Perú S.A.C.\n\nGESTIÓN DE CONTRASEÑAS\nTodas las contraseñas de sistemas críticos deben cumplir:\n- Mínimo 16 caracteres\n- Al menos 1 mayúscula, 1 número y 1 carácter especial\n- Rotación obligatoria cada 90 días\n- No reutilizar las últimas 12 contraseñas\nEl gestor de contraseñas corporativo es 1Password Business. Prohibido guardar contraseñas en texto plano.\n\nACCESO A SISTEMAS\nEl acceso a servidores de producción requiere:\n1. Aprobación del Tech Lead del área\n2. Autenticación de dos factores (2FA) obligatoria\n3. Acceso únicamente mediante VPN corporativa (WireGuard)\n4. Registro en el sistema de auditoría SIEM\nLos accesos no usados por más de 60 días se revocan automáticamente.\n\nPOLÍTICA DE DATOS\nLos datos de clientes son confidenciales nivel 3. No pueden salir de los servidores corporativos sin cifrado. Violaciones se reportan al DPO en 24 horas.",

    "manual_deploy.txt": "PROCESO DE DEPLOY — TechCorp Perú S.A.C.\n\nAMBIENTES\nTechCorp mantiene 4 ambientes:\n- desarrollo (dev): rama feature/*, deploy automático por commit\n- integración (staging): rama develop, deploy automático por merge\n- pre-producción (pre-prod): rama release/*, deploy manual con aprobación\n- producción (prod): rama main, deploy manual con aprobación de 2 revisores\n\nVENTANAS DE DEPLOY A PRODUCCIÓN\nLos deploys a producción solo se permiten:\n- Martes y jueves entre 10:00 AM y 12:00 PM hora Lima\n- Nunca en viernes, fines de semana ni feriados\n- En caso de hotfix crítico: cualquier día con aprobación del CTO\n\nROLLBACK\nSi un deploy causa incidentes:\n1. Detectar el problema (Grafana o reporte de usuario)\n2. Notificar al canal #incidentes en Slack\n3. Ejecutar: kubectl rollout undo deployment/app-prod\n4. Verificar que el servicio se restauró\n5. Crear un post-mortem dentro de las 24 horas siguientes\nLa aprobación del rollback la da el Tech Lead de turno.",

    "manual_monitoreo.txt": "MONITOREO Y ALERTAS — TechCorp Perú S.A.C.\n\nSTACK DE MONITOREO\n- Métricas: Prometheus + Grafana (dashboard: grafana.techcorp.pe)\n- Logs: ELK Stack (Elasticsearch, Logstash, Kibana)\n- Trazabilidad: Jaeger para distributed tracing\n- Alertas: PagerDuty para incidentes P1 y P2\n\nNIVELES DE INCIDENTE\n- P1 (Crítico): sistema caído, pérdida de datos, brecha de seguridad. Tiempo de respuesta: 15 minutos. Responsable: DevOps on-call + CTO notificado automáticamente.\n- P2 (Alto): degradación significativa. Tiempo de respuesta: 1 hora.\n- P3 (Medio): funcionalidad no crítica afectada. Respuesta en 4 horas.\n- P4 (Bajo): mejoras y problemas menores. Respuesta en 1 semana.\n\nPOST-MORTEM\nTodo incidente P1 o P2 requiere un post-mortem en Confluence dentro de las 48 horas siguientes al cierre.",

    "manual_rrhh.txt": "POLÍTICAS DE RECURSOS HUMANOS — TechCorp Perú S.A.C.\n\nVACACIONES\nCada colaborador tiene derecho a 30 días calendario de vacaciones al año. Las vacaciones se solicitan con al menos 2 semanas de anticipación. No se pueden tomar más de 15 días consecutivos sin aprobación del gerente.\n\nTRABAJO REMOTO\nEl esquema es híbrido: 3 días presencial y 2 remoto por semana. Los días presenciales obligatorios son martes, miércoles y jueves.\n\nCAPACITACIONES\nCada colaborador tiene un presupuesto anual de S/. 2,000 para capacitaciones. TechCorp tiene convenios con Platzi, Coursera y Udemy Business.\n\nEVALUACIÓN DE DESEMPEÑO\nLas evaluaciones se realizan semestralmente: junio y diciembre. El sistema de calificación es del 1 al 5. Una calificación de 4 o 5 hace al colaborador elegible para bono.",
}

os.makedirs("./docs_techcorp", exist_ok=True)
for nombre, contenido in documentos.items():
    with open(f"./docs_techcorp/{nombre}", "w", encoding="utf-8") as f:
        f.write(contenido)

print(f"Colección creada: {len(documentos)} documentos")
for nombre, contenido in documentos.items():
    print(f"  {nombre}: {len(contenido.split())} palabras")

Colección creada: 5 documentos
  manual_servidores.txt: 155 palabras
  manual_seguridad.txt: 139 palabras
  manual_deploy.txt: 152 palabras
  manual_monitoreo.txt: 115 palabras
  manual_rrhh.txt: 121 palabras


---
## Parte 3 — Ingesta y chunking

Con 5 documentos no podemos enviarlos todos en cada consulta. Los dividimos en chunks pequeños para recuperar solo los relevantes para cada pregunta.

In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
import glob
from collections import Counter

# cargar todos los archivos guardando la fuente en metadata
todos_los_docs = []
for ruta in sorted(glob.glob("./docs_techcorp/*.txt")):
    nombre = ruta.split("/")[-1]
    with open(ruta, "r", encoding="utf-8") as f:
        contenido = f.read()
    todos_los_docs.append(Document(page_content=contenido, metadata={"fuente": nombre}))

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)
chunks = splitter.split_documents(todos_los_docs)

print(f"Documentos cargados: {len(todos_los_docs)}")
print(f"Total chunks: {len(chunks)}")
print()
for doc, n in sorted(Counter(c.metadata["fuente"] for c in chunks).items()):
    print(f"  {doc}: {n} chunks")

Documentos cargados: 5
Total chunks: 13

  docs_techcorp\manual_deploy.txt: 3 chunks
  docs_techcorp\manual_monitoreo.txt: 3 chunks
  docs_techcorp\manual_rrhh.txt: 2 chunks
  docs_techcorp\manual_seguridad.txt: 2 chunks
  docs_techcorp\manual_servidores.txt: 3 chunks


---
## Parte 4 — Embeddings y base vectorial

In [4]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

print("Cargando modelo de embedding (puede tardar la primera vez)...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vector_prueba = embeddings.embed_query("prueba")
print(f"Dimensiones del embedding: {len(vector_prueba)}")

print("Indexando chunks en ChromaDB...")
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_lab04",
)
print(f"Documentos indexados: {vectorstore._collection.count()}")

Cargando modelo de embedding (puede tardar la primera vez)...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\djace\ai-developer-2026\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\djace\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Dimensiones del embedding: 384
Indexando chunks en ChromaDB...
Documentos indexados: 13


---
## Parte 5 — Retrieval entre múltiples documentos

El retriever distingue qué documento contiene la respuesta para cada pregunta, sin que se lo indiquemos.

In [5]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# cada pregunta apunta a un documento distinto
preguntas_test = [
    "¿Cuándo se pueden hacer deploys a producción?",
    "¿Cuáles son los requisitos de las contraseñas?",
    "¿Cuántos días de vacaciones tienen los empleados?",
    "¿Qué herramienta se usa para distributed tracing?",
]

for pregunta in preguntas_test:
    docs_recuperados = retriever.invoke(pregunta)
    print(f"Pregunta: '{pregunta}'")
    for doc in docs_recuperados:
        print(f"  [{doc.metadata['fuente']}] '{doc.page_content[:80]}...'")
    print()

Pregunta: '¿Cuándo se pueden hacer deploys a producción?'
  [docs_techcorp\manual_deploy.txt] 'VENTANAS DE DEPLOY A PRODUCCIÓN
Los deploys a producción solo se permiten:
- Mar...'
  [docs_techcorp\manual_servidores.txt] 'SERVIDORES DE PRODUCCIÓN
TechCorp opera 3 servidores de producción en el datacen...'
  [docs_techcorp\manual_seguridad.txt] 'POLÍTICAS DE SEGURIDAD — TechCorp Perú S.A.C.

GESTIÓN DE CONTRASEÑAS
Todas las ...'

Pregunta: '¿Cuáles son los requisitos de las contraseñas?'
  [docs_techcorp\manual_seguridad.txt] 'POLÍTICAS DE SEGURIDAD — TechCorp Perú S.A.C.

GESTIÓN DE CONTRASEÑAS
Todas las ...'
  [docs_techcorp\manual_rrhh.txt] 'POLÍTICAS DE RECURSOS HUMANOS — TechCorp Perú S.A.C.

VACACIONES
Cada colaborado...'
  [docs_techcorp\manual_deploy.txt] 'VENTANAS DE DEPLOY A PRODUCCIÓN
Los deploys a producción solo se permiten:
- Mar...'

Pregunta: '¿Cuántos días de vacaciones tienen los empleados?'
  [docs_techcorp\manual_rrhh.txt] 'POLÍTICAS DE RECURSOS HUMANOS — TechCorp Per

---
## Parte 6 — Pipeline RAG completo

In [6]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0.1, max_tokens=500)

PROMPT_RAG = ChatPromptTemplate.from_template("""
Eres un asistente técnico de TechCorp. Responde usando ÚNICAMENTE el contexto dado.
Si la respuesta no está en el contexto, di: "No encuentro esa información en el manual."

Contexto:
{contexto}

Pregunta: {pregunta}
Respuesta:
""")

pipeline_rag = (
    {"contexto": retriever | (lambda docs: "\n\n".join(d.page_content for d in docs)),
     "pregunta": RunnablePassthrough()}
    | PROMPT_RAG
    | llm
    | StrOutputParser()
)

print("Pipeline RAG listo")

Pipeline RAG listo


In [7]:
preguntas = [
    "¿En qué días y horarios se pueden hacer deploys a producción?",
    "¿Cuántos caracteres mínimos debe tener una contraseña?",
    "¿Cuántos días de vacaciones anuales tienen los empleados?",
    "¿Qué pasa si se detecta un incidente P1?",
    # cruza dos documentos: deploy + monitoreo
    "¿Qué canal se usa para reportar problemas de deploy y cuánto tiempo hay para el post-mortem?",
    # no está en ningún documento
    "¿Cuál es el salario promedio de los ingenieros de TechCorp?",
]

for pregunta in preguntas:
    print(f"Pregunta: {pregunta}")
    print(f"Respuesta: {pipeline_rag.invoke(pregunta)}")
    print()

Pregunta: ¿En qué días y horarios se pueden hacer deploys a producción?
Respuesta: Los deploys a producción en TechCorp se permiten en:

- **Martes y jueves entre 10:00 AM y 12:00 PM** (hora Lima)
- **Nunca en viernes, fines de semana ni feriados**
- **Excepción:** En caso de hotfix crítico, se permite cualquier día con aprobación del CTO

Pregunta: ¿Cuántos caracteres mínimos debe tener una contraseña?
Respuesta: Según las Políticas de Seguridad de TechCorp, una contraseña de sistemas críticos debe tener un **mínimo de 16 caracteres**.

Pregunta: ¿Cuántos días de vacaciones anuales tienen los empleados?
Respuesta: Cada colaborador tiene derecho a **30 días calendario de vacaciones al año**.

Pregunta: ¿Qué pasa si se detecta un incidente P1?
Respuesta: Si se detecta un incidente P1:

- **Clasificación**: Es un incidente Crítico (sistema caído, pérdida de datos, o brecha de seguridad)
- **Tiempo de respuesta**: 15 minutos
- **Responsables**: DevOps on-call + CTO notificado automáticame

---
## Parte 7 — Con RAG vs sin RAG

In [8]:
import anthropic
cliente = anthropic.Anthropic()

def responder_sin_rag(pregunta):
    r = cliente.messages.create(
        model="claude-haiku-4-5-20251001",
        system="Eres un asistente técnico de TechCorp.",
        messages=[{"role": "user", "content": pregunta}],
        max_tokens=300, temperature=0.1,
    )
    return r.content[0].text


# pregunta que cruza dos documentos — imposible responder correctamente sin RAG
PREGUNTA = "¿En qué días puede hacerse un hotfix en TechCorp, quién lo aprueba y a qué canal se reporta?"

print("SIN RAG:")
print(responder_sin_rag(PREGUNTA))
print()
print("CON RAG:")
print(pipeline_rag.invoke(PREGUNTA))
print()
print("La respuesta correcta requiere manual_deploy.txt Y manual_monitoreo.txt.")

SIN RAG:
# Política de Hotfix en TechCorp

Basándome en la información que tengo como asistente técnico de TechCorp, debo ser honesto: **no tengo acceso a la política específica de hotfixes** en mi base de conocimiento actual.

Para obtener esta información precisa, te recomiendo:

1. **Consultar la documentación oficial** de TechCorp sobre procesos de release
2. **Contactar a tu equipo de DevOps/Release Management** - ellos manejan estas políticas
3. **Revisar el wiki o repositorio interno** de procedimientos
4. **Hablar con tu líder técnico o manager** que conoce los protocolos vigentes

Estos detalles (días permitidos, aprobadores, canales de reporte) son críticos y pueden variar según:
- El ambiente (producción, staging, etc.)
- El tipo de hotfix
- Las políticas actuales de la empresa

¿Hay algo más específico sobre procesos técnicos en los que pueda ayudarte?

CON RAG:
Según el manual:

**Días para hotfix crítico:** Cualquier día de la semana

**Aprobación requerida:** CTO (Chief 

---
## Parte 8 — Evaluación básica

In [9]:
eval_set = [
    {"pregunta": "¿Cuántos servidores de producción tiene TechCorp?",      "clave": "3"},
    {"pregunta": "¿Cada cuánto tiempo rotar las contraseñas?",              "clave": "90"},
    {"pregunta": "¿Qué gestor de contraseñas usa la empresa?",              "clave": "1Password"},
    {"pregunta": "¿Cuál es el comando de rollback en kubernetes?",           "clave": "kubectl rollout undo"},
    {"pregunta": "¿Qué herramienta usa TechCorp para distributed tracing?",  "clave": "Jaeger"},
]

aciertos = 0
for item in eval_set:
    respuesta = pipeline_rag.invoke(item["pregunta"])
    correcto = item["clave"].lower() in respuesta.lower()
    if correcto:
        aciertos += 1
    print(f"{'OK' if correcto else 'FAIL'} | {item['pregunta'][:55]}")

print(f"\nAccuracy: {aciertos}/{len(eval_set)} = {aciertos/len(eval_set)*100:.0f}%")

OK | ¿Cuántos servidores de producción tiene TechCorp?
OK | ¿Cada cuánto tiempo rotar las contraseñas?
OK | ¿Qué gestor de contraseñas usa la empresa?
OK | ¿Cuál es el comando de rollback en kubernetes?
OK | ¿Qué herramienta usa TechCorp para distributed tracing?

Accuracy: 5/5 = 100%
